In [ ]:
from __future__ import division
import numpy as np
import scipy
import time
import matplotlib.pyplot as plt
plt.rcParams.update( plt.rcParamsDefault )
import warnings
warnings.filterwarnings( 'ignore')
np.random.seed(1234)
plt.rcParams.update( { 'font.size' : 12 } )
%matplotlib inline
%load_ext autoreload
%autoreload 

In [ ]:
import os
relative_path_to_new_folder = "../Images"
os.makedirs( relative_path_to_new_folder, exist_ok = True )
image_folder_path = relative_path_to_new_folder + "/Iterative_optimization_images"
if not os.path.isdir( image_folder_path ):
    os.makedirs( image_folder_path )

# Entropy regularized formulation

The primal entropy regularized formulation of OT is given by:
$$
OT_{\epsilon}(\alpha,\beta) = min_{\pi \in \mathcal{U}(\alpha,\beta)} \langle C,\pi \rangle +\epsilon KL(\pi\|\alpha \otimes \beta)\ ,
$$
where
$\ 
KL(\pi\|\alpha \otimes \beta) 
\ $ is the KL-divergence and $\ \mathcal{U}(\alpha,\beta)=\{\pi: \pi\mathcal{1}=\alpha\ , \pi^{T}\mathcal{1}=\beta\}\ .$

The optimal coupling $\pi^{*}$ has the following form :
$$
\pi^{*} = \alpha \odot \Delta(u)K \Delta(v)\odot \beta\ ,\ \Delta: diag: \R^{n} \mapsto M_{n}\left(\R\right)
$$
and we know that $\pi^{*}\mathbb{1}=\alpha$ and $(\pi^{*})^{T}\mathbb{1}=\beta\ .$
# I. Sinkhorn
Therefore, Sinkhorn updates is given by the following alternative projections
$$
u^{(t+1)}  \leftarrow \frac{1}{K(v^{(t)}\odot \beta)}\ ,\ 
v^{(t+1)}  \leftarrow \frac{1}{K^{T}(u^{(t+1)}\odot \alpha)}\ , 
$$
where 
$K = \exp\left(-\frac{C}{\epsilon}\right)\in M_{n\times m}(\mathbb{R}),\ \alpha \in \mathbb{R}^{n},\ \beta \in \mathbb{R}^{m}\ ,\ u\in \mathbb{R}^{n},\ v\in \mathbb{R}^{m}\ $ and $ \ (u^{(0)},v^{(0)})=(u,v)\ .$
The denominator in these iterates $K(v^{(t)}\odot \beta)$ and $K^{T}(u^{(t+1)}\odot \alpha)$ are sum of exponentials which are unregularized and will causes both overflow and underflow causing the algorithm to terminate without convergence.


In [ ]:
class sinkhorn:
  def __init__( self, K, a, b, u, v, epsilon ):
    """

    Parameters:
    -----------
        K : ndarray, shape (n,m),
            The Gibb's kernel.
        a : ndarray, shape (n,),
            The probability histogram of the sample of size n.
        b : ndarray, shape (m,),
            The probability histogram of the sample of size m.
        u : ndarray, shape (n,),
            The initial u.
        v : ndarray, shape (m,),
            The initial v.
        epsilon : float,
                  The regularization parameter.
    """
    self.K = K
    self.a = a
    self.b = b
    self.u = u
    self.v = v
    self.epsilon = epsilon
    self.errors = []
    self.objective_values = []
    
  def _objectivefunction( self ):
    """
    
    Returns:
    --------
      Q(f,g)  : float,
                The value of objective function obtained by evaluating the formula Q(f,g) = < f, a > + < g, b > - epsilon*< u, Kv >,
                where u = exp( f/epsilon ), v = exp( g/epsilon ). 
    """
    f = np.log( self.u ) * self.epsilon
    g = np.log( self.v ) * self.epsilon
    target = np.dot( f, self.a ) + np.dot( g, self.b )
    penalization = -self.epsilon*np.dot( np.exp( f/self.epsilon ).T, np.dot( self.K, np.exp( g/self.epsilon ) ) )
    return target + penalization

  def _optimize( self, tol = 1e-12, max_iterations = 1000 ):
    """

    Parameters:
    -----------
      tol : float,
            The toleranc for the error. Defaults to 1e-12.
      max_iterations  : int,
                        The maximum iteration for the optimization algorithm. Defaults to 1000.

    Returns:
    --------
      Returns a dictionary where the keys are strings and the values are ndarrays or list.
      The following are the keys of the dictionary and the descriptions of their values:
        potential_f : ndarray, shape (n,),
                      The optimal Kantorovich potential f.
        potential_g : ndarray, shape (m,),
                      The optimal Kantorovich potential g.
        errors   :   list,
                        The list of errors observed when checking conservation of mass .
        objective_values : list,
                           The list of objective values observed after each ascent update.
    """
    i = 1
    while True :
      # Projection 1
      self.u = self.a / np.dot( self.K, self.v )
      r = self.v * np.dot( self.K.T, self.u)
      # Projection 2
      self.v = self.b / np.dot( self.K.T, self.u )
      # Check conservation of mass
      s = self.u * np.dot( self.K, self.v )
      # Check conservation of mass: ||P1_m - a||_1 + ||P^T 1_n - b||_1
      self.errors.append(    np.linalg.norm( r - self.b )
                            + 
                            np.linalg.norm( s - self.a ) 
                        )
      self.objective_values.append( self._objectivefunction() )
      condition = ( self.errors[-1] > tol )
      if i < max_iterations and condition :
          i += 1
      elif np.isnan( self.errors[-1] ):
        print( "Terminating at iteration: ", i,  ", due to underflow and overflow of values while performing the alternative projection. " )
        break
      else:
        print( "Terminating after iteration: ", i  )
        break   
    # end for
    return {
      'potential_f'       : self.epsilon * np.log( self.u ),
      'potential_g'       : self.epsilon * np.log( self.v ),
      'errors'            : self.errors,
      'objective_values'  : self.objective_values
    }


# II. Log-domain Sinkhorn

The log-exp regularized update of the Sinkhorn algorithm is given by
$$
m_{i}(g^{(t)})\leftarrow \max_{j}( g^{(t)}_{j} - C_{ij} )\ ,\ \forall\  i = 1,\dots,n\ ,
$$
$$
f^{(t+1)}_{i}\leftarrow -\varepsilon \log\left(\sum_{j=1}^{m}\exp\left(\frac{\left( g_{j}^{(t)} - C_{ij}-m_{i}(g^{(t)})\right)}{\varepsilon}\right)\beta_{j}\right)-m_{i}(g^{(t)})\ ,\ \forall\  i=1,\dots,n\ ,
$$
$$
m_{j}(f^{(t+1)})\leftarrow \max_{i}( f^{(t+1)}_{i} - C_{ij} )\ ,\ \forall\   j=1,\dots,m\
 ,
$$
$$
g^{(t+1)}_{j}\leftarrow -\varepsilon \log\left(\sum_{i=1}^{n}\exp\left(\frac{\left( f_{i}^{(t+1)} - C_{ij} - m_{j}(f^{(t+1)})\right)}{\varepsilon}\right)\alpha_{i}\right) - m_{j}(f^{(t+1)})\ ,\ \forall\  j=1,\dots,m\ ,
$$
where 
$\varepsilon >0,\ $ $\alpha \in \mathbb{R}^{n},\ $ $\beta \in \mathbb{R}^{m},\ $
   $f \in \mathbb{R}^{n},\ $ $g \in \mathbb{R}^{m}\ $ and $\ (f^{(0)},g^{(0)})=(f,g)\ .$

This regularization ensures that the summation inside the log is always in $(0,1]$, which ensures immunity to underflow and overflow of values.

In [ ]:
class log_domainSinkhorn:
    def __init__( self, a, b, C, epsilon ):
        """
        
        Parameters:
        -----------
            C   :   ndarray, shape (n,m), 
                    It is the cost matrix between the points sampled from the point clouds.
            a   :   ndarray, shape (n,),
                    The probability histogram of the sample of size n.
            b   :   ndarray, shape (m,),
                    The probability histogram of the sample of size m.
            epsilon     :   float,
                            The regularization parameter.
        """
        self.a       = a
        self.b       = b
        self.C       = C
        self.epsilon = epsilon
        self.errors   = []
    
    def _f( self, H ):
        """
        Here we compute the value of the potential f by using its Schrodinger-bridge relation with the potential g: - epsilon * log( ( b * exp( H/epsilon ) )1_{m} ).
        Parameters:
        -----------
            H   :   ndarray, shape (n,m),
                    It is the matrix obtained from the difference g - C.
        Returns:
        --------
            ndarray, shape (n,),
            The value of potential f.
        """
        return - self.epsilon * np.log( np.sum( self.b[None,:] * np.exp( H/self.epsilon ), axis = 1 ) )# Shape: (n,)
    
    def _logexp_f( self, H ):
        """
        Here we incorporate the log-exp regularization method in the computation of the potential f.
        Parameters:
        -----------
            H   :   ndarray, shape (n,m),
                    It is the matrix obtained from the difference g - C.

        Returns:
        --------
            ndarray, shape (n,),
            The log-exp regularized value of potential f.

        """
        return self._f( H - np.max( H, axis = 1 )[:,None] ) - np.max( H, axis = 1 )# Shape: (n,)


    def _g( self, H ):
        """
        Here we compute the value of the potential g by using its Schrodinger-bridge relation with the potential f: - epsilon * log( 1_{n}^{T} ( a * exp( H/epsilon ) ) ).
        Parameters:
        -----------
            H   :   ndarray, shape (n,m),
                    It is the matrix obtained from the difference f - C.
        Returns:
        --------
            ndarray, shape (m,),
            The value of potential g.
        """
        return - self.epsilon * np.log( np.sum( self.a[:,None] * np.exp( H/self.epsilon ), axis = 0 ) )# Shape: (m,)
    
    def _logexp_g( self, H ):
        """
        Here we incorporate the log-exp regularization method in the computation of the potential g.
        Parameters:
        -----------
            H   :   ndarray, shape (n,m),
                    It is the matrix obtained from the difference f - C.

        Returns:
        --------
            ndarray, shape (m,),
            The log-exp regularized value of potential g.

        """
        return self._g( H - np.max( H, axis = 0 )[None,:] )  - np.max( H, axis = 0 )# Shape: (m,)

    
    def _optimize( self, tol = 1e-12, max_iterations = 500 ):     
        """
        
        Parameters:
        -----------
            tol     :   float,
                        The tolerance for the error. Defaults to 1e-12.
            max_iterations      :   int,
                                    The maximum iteration for the optimization algorithm. Defaults to 500.
        Returns:
        --------
        Returns a dictionary where the keys are strings and the values are ndarrays or list.
        The following are the keys of the dictionary and the descriptions of their values:
            potential_f     :   ndarray, shape (n,),
                                The optimal Kantorovich potential f.
            potential_g     :   ndarray, shape (m,),
                                The optimal Kantorovich potential g.
            error   :   list,
                        The list of errors observed when checking conservation of mass .
            iterations  :   int,
                            Total number of iterations performed.
        """
        f = self.a
        for i in range( max_iterations ):
            g = self._logexp_g( f[:,None] - self.C )# Shape: (m,)
            f = self._logexp_f( g[None,:] - self.C )# Shape: (n,)
            # Computing the coupling matrix
            P = self.a[:,None] * np.exp( ( f[:,None] + g[None,:]  - self.C )/self.epsilon ) * self.b[None,:]# Shape: (n,m),  line (*)
            # Check conservation of mass: ||P1_m - a||_1 + ||P^T 1_n - b||_1
            self.errors.append(  np.linalg.norm( np.sum( P, axis = 1 ) - self.a, ord = 1 )
                                +
                                np.linalg.norm( np.sum( P, axis = 0 ) - self.b, ord = 1 )
                             )
            if self.errors[-1] < tol:
                break
            if i + 1 >= max_iterations:
                break
        # Change of convention because of line (*)
        f = f + self.epsilon * np.log( self.a )
        g = g + self.epsilon * np.log( self.b )
        # end for
        return {
            'errors'            : self.errors,                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      
            'potential_f'       : f,
            'potential_g'       : g, 
            'iterations'        : i + 1
        }

# III. Iterative optimization

The following class in the semi-dual framework perform update of the potentials by comparing the updates of log-domain Sinkhorn ( described above ) and damped Newton based on the increment in the objective function value.


## Dual of Entropic formulation
The dual of the entropic formulation of the OT is given by

\begin{align*}
\max_{(f,g)\in \R^{n}\times\R^{m} }Q_{C,\alpha,\beta,\varepsilon}(f,g):=\langle f,\ \alpha \rangle 
		+ \langle g,\ \beta \rangle
		-\varepsilon 
\left(\langle \exp\left(\frac{f+g-C}{\varepsilon}\right),\ \alpha\otimes \beta\rangle-1\right)\ ,
\end{align*}	
where $\odot$ denotes the elementwise product.

### Static Schrodinger bridge relations
The optimal potentials $(f^{*},g^{*})$ are given by the relations
\begin{align*}
f^{*}_{i} =  - \varepsilon\log\left(\sum_{j}^{m}\exp\left(\frac{g^{*}_{j}-C_{ij}}{\varepsilon}\right)\beta_{j}\right),\ \forall\ i = 1.\dots,n\ ,
\end{align*}

\begin{align*}
g^{*}_{j}= - \varepsilon\log\left(\sum_{i=1}^{n}\exp\left(\frac{f^{*}_{i}-C_{ij}}{\varepsilon}\right)\alpha_{i}\right),\ \forall\ j = 1,\dots,m\ .
\end{align*}
 
## Semi dual formulation
The semi-dual formulation is obtained by plugging-in one of the static Schrodinger-bridge relations between the two potentials in the dual of the entropy regularized formulation at the optimum.
Therefore, on plugging-in the relation of $g$ with $f$ we have
\begin{align*}
Q^{semi}_{C,\alpha,\beta,\varepsilon}(f) & := \langle f,\ \alpha \rangle 
		+ \langle g(f,C,\varepsilon),\ \beta \rangle
		-\varepsilon 
\left(\langle  \exp\left(\frac{f+g(f,C,\varepsilon)-C}{\varepsilon}\right),\ \alpha\otimes \beta\rangle-1\right)\\
& = \langle f,\ \alpha \rangle 
		+ \langle g(f,C,\varepsilon),\ \beta \rangle
		-\varepsilon 
\left( 1-1\right)\\
& = \langle f,\ \alpha \rangle + \langle g(f,C,\varepsilon),\ \beta \rangle\ , 
\end{align*}
where
$g(f,C,\varepsilon)_{j} := -\varepsilon\log\left(\sum_{i}\exp\left(\frac{f_{i}-C_{ij}}{\varepsilon}\right)\alpha_{i}\right)\ ,\ \forall j = 1,\dots,m\ .$

Let us look at the gradients and the Hessian of this formulation.

Let 
\begin{align*}\Gamma^{C,\alpha}(f,\varepsilon)_{ij} := \frac{\alpha_{i}\exp\left(\frac{f_{i} - C_{ij}}{\varepsilon}\right)}{\left(\sum_{i=1}^{n}\alpha_{i}\exp\left(\frac{f_{i} - C_{ij}}{\varepsilon}\right)\right)}\ ,\ \forall\ i = 1,\dots, n\ ,\ j = 1,\dots, m\ .
\end{align*}
### Gradients

\begin{align*}
\nabla_{f}Q^{semi}_{C,\alpha,\beta,\varepsilon}(f)_{i} &= \alpha_{i}-\sum_{j=1}^{m}\Gamma^{C,\alpha}(f,\varepsilon)_{ij}\beta_{j}\ ,\ \forall\ i = 1,\dots,n\ .
\end{align*}

### Hessian 

\begin{align*}
\nabla_{f}^{2}Q^{semi}_{C,\alpha,\beta,\varepsilon}(f) = \frac{-1}{\varepsilon}\left(\Delta( ( \Gamma^{C,\alpha}(f,\varepsilon)\odot \beta )\mathcal{1}_{m} )- \Gamma^{C,\alpha}(f,\varepsilon)\Delta(\beta)( \Gamma^{C,\alpha}(f,\varepsilon)^{T}\ )\right) ,
\end{align*}
where $\odot$ is the elementwise multiplication and $\Delta: diag: \R^{n} \mapsto M_{n}(\R)\ .$

### Log-exp regularization
The log-exp regularization of $g(f,C,\varepsilon)$ is the same as that in log-domain Sinkhorn while the regularization of the gradients and the Hessian can be done by regularizing the matrix $\Gamma^{C,\alpha}(f,\varepsilon)$ in the following way
\begin{align*}
\Gamma^{C,\alpha}(f,\varepsilon)_{ij} = \frac{\alpha_{i}\exp\left(\frac{f_{i} - C_{ij} - m_{j}(f)}{\varepsilon}\right)}{\left( \sum_{i=1}^{n}\alpha_{i}\exp\left(\frac{f_{i} - C_{ij} - m_{j}(f)}{\varepsilon}\right)\right)}\ ,\ \forall\ i = 1, \dots,n\ ,\ j = 1,\dots,m\ ,
\end{align*}
where $m_{j}(f) := \max_{i}( f_{i} - C_{ij} )\ ,\ \forall\ j = 1,\dots,m\ .$ Note that this regularization ensures that there are no overflows or underflow in the denominator in the entries of $\Gamma^{C,\alpha}(f,\varepsilon)$ but there can be elements with underflow in the numerator when $\varepsilon$ is small.

### Damped Newton 
#### Preconditioned inversion of the Hessian 

Let $H^{C,\alpha, \beta}(f,\varepsilon)$ denote the unnormalized Hessian, that is, $H^{C,\alpha, \beta}(f,\varepsilon) := - \varepsilon \nabla^{2}_{f}Q^{semi}_{C,\alpha,\beta,\varepsilon}(f)\ .$

By Perron-Frobenius theorem $\dim\left(\Delta\left(\frac{1}{\sqrt{\alpha}}\right)H^{C,\alpha, \beta}(f,\varepsilon)\Delta\left(\frac{1}{\sqrt{\alpha}}\right)\right) = 1 $ and $\mathcal{1}_{n}\in \ker\left(\Delta\left(\frac{1}{\sqrt{\alpha}}\right)H^{C,\alpha, \beta}(f,\varepsilon)\Delta\left(\frac{1}{\sqrt{\alpha}}\right)\right)\ .$ Therefore, the Hessian is not invertible. Also if we observve the spectrum of the Hessian we find that as $\varepsilon$ decreases more eigenvalues clutter near zero. 

To ensure the stability of inverting the Hessian during damped Newton we precondition it in two stages:
##### i. Null vector preconditioning:
Here we precondition the Hessian with the null vector to make sure it is invertible.
\begin{align*}
    \tilde{H}^{C,\alpha, \beta}(f,\varepsilon)  := \Delta\left(\frac{1}{\sqrt{\alpha}}\right)H^{C,\alpha, \beta}(f,\varepsilon)\Delta\left(\frac{1}{\sqrt{\alpha}}\right) + \tau \mathcal{1}_{n}\mathcal{1}_{n}^{T}\ .
\end{align*}
##### ii. True preconditioning:
Here we move $k\leq n$ non-zero eigenvalues that are close to 0 to 1 which ensures stability of the inversion. The preconditioning matrix is given by
\begin{align*}
P := I_{n} + \sum_{i=1}^{k}\left(\frac{1}{\sqrt{\lambda}}-1\right)y_{i}y_{i}^{T}\ ,
\end{align*}
where $y_{i}\in \ker\left(\Delta\left(\frac{1}{\sqrt{\alpha}}\right)\tilde{H}^{C,\alpha, \beta}(f,\varepsilon)\Delta\left(\frac{1}{\sqrt{\alpha}}\right)-\lambda_{i}I_{n}\right)\ ,\ \langle y_{i}, y_{j}\rangle = \delta_{ij}\ $
such that
\begin{align*}
P\Delta\left(\frac{1}{\sqrt{\alpha}}\right)\tilde{H}^{C,\alpha, \beta}(f,\varepsilon)\Delta\left(\frac{1}{\sqrt{\alpha}}\right)Py_{i} = y_{i}\ ,\ \forall\ i = 1,\dots,k\ ,
\end{align*}
where $\delta_{ij}$ is the Kronecker delta.

For inversion we use either use exact or iterative methods of inversion such as conjugate gradient or GMRES.
#### Line search

For $\alpha = 1,\ c,\ \rho \ \in\  (0,1)\ ,$

while $Q^{semi}_{C,\alpha,\beta,\varepsilon}( f + \alpha p ) < Q^{semi}_{C,\alpha,\beta,\varepsilon}( f ) + \alpha c \langle p, \nabla_{f}Q^{semi}_{C,\alpha,\beta,\varepsilon}(f) \rangle: $
$$
\alpha \leftarrow \rho \alpha\ ,
$$
where $\alpha$ is the update step size, $c$ is the sufficient increase parameter and $\rho$ is the damping factor.



In [ ]:
class iterative_optimization:
    def __init__( self, C, a, b, f, epsilon, rho, c, null_vector, precond_vectors ):
        """

        Parameters:
        ----------- 
            C   :   ndarray, shape (n,m), 
                    It is the cost matrix between the points sampled from the point clouds.
            a   :   ndarray, shape (n,),
                    The probability histogram of the sample of size n.
            b   :   ndarray, shape (m,),
                    The probability histogram of the sample of size m.
            f   :   ndarray, shape (n,), 
                    The initial Kantorovich potential f.
            rho     :   float,
                        Damping factor for the line search ascent step size.
            c   :   float,
                    Sufficient increase parameter in the Armijo condition .
            epsilon     :   float,
                            The regularization parameter.
            null_vector     :   ndarray, shape (n,),
                                The null vector of the Hessian to be used for null vector preconditioning .
            precond_vectors     :   list of ndarrays, shape (n,),
                                    The stack of preconditioning vectors obtained from the Hessian at the optimum obtained from the algorithm without any preconditioning,
                                    that is, semi-dual damped Newton with only null vector preconditioning and exact inversion..
        """
        self.C = C
        self.a = a
        self.b = b
        self.f = f
        self.epsilon = epsilon
        self.rho = rho           
        self.c = c
        self.null_vector = null_vector
        self.precond_vectors = precond_vectors
        self.alpha_list = []
        self.errors = []
        self.objective_values = [] 
        self.update_indicator = {}
        
    def _objectivefunction( self, f, g ) :
        """ 
        Parameters:
        -----------
            f   :   ndarray, shape (n,),
                    The input Kantorovich potential f.
            g   :   ndarray, shape (m,),
                    The input Kantorovich potential g.
                
        Returns: 
        --------
            Q_semi(f)   :   float
                            The value of semi-dual objective function obtained by evaluating Q_semi(f) = < f, a > + < g( f, C, epsilon ), b >,
                            where g( f, C, epsilon ) denotes the value of Kantorovich potential g evaluated using the Schrodinger-bridge equations between f and g.
        """
        Q_semi = np.dot( f, self.a ) + np.dot( g, self.b )
        return Q_semi

    def _computegradientf( self ):
        """ 
            Here we compute the gradient of the objective function at the current value of potential f.
            Returns:
            --------
            ndarray, shape (n,).
        """

        gradient =  self.a - np.sum( self.normalized_exp * self.b[None,:], axis = 1 ) # Shape: (n,)
        return gradient
    
    def _f( self, H ):
        """
        Here we compute the value of the potential f by using its Schrodinger-bridge relation with the potential g: - epsilon * log(  ( b * exp( H/epsilon ) )1_{m} ).
        Parameters:
        -----------
            H   :   ndarray, shape (n,m),
                    It is the matrix obtained from the difference g - C.
        Returns:
        --------
            ndarray, shape (n,),
            The value of potential f.
        """
        return - self.epsilon * np.log( np.sum( self.b[None,:] * np.exp( H/self.epsilon ), axis = 1 ) )# Shape: (n,)
    
    def _logexp_f( self, H ):
        """
        Here we incorporate the log-exp regularization method in the computation of the potential f.
        Parameters:
        -----------
            H   :   ndarray, shape (n,m),
                    It is the matrix obtained from the difference g - C.

        Returns:
        --------
            ndarray, shape (n,),
            The log-exp regularized value of potential f.

        """
        return self._f( H - np.max( H, axis = 1 )[:,None] ) - np.max( H, axis = 1 )# Shape: (n,)


    def _g( self, H ):
        """
        Here we compute the value of the potential g by using its Schrodinger-bridge relation with the potential f: - epsilon * log( 1_{n}^{T} ( a * exp( H/epsilon ) ) ).
        Parameters:
        -----------
            H   :   ndarray, shape (n,m),
                    It is the matrix obtained from the difference f - C.
        Returns:
        --------
            ndarray, shape (m,),
            The value of potential g.
        """
        return - self.epsilon * np.log( np.sum( self.a[:,None] * np.exp( H/self.epsilon ), axis = 0 ) )# Shape: (m,)
    
    def _logexp_g( self, H ):
        """
        Here we incorporate the log-exp regularization method in the computation of the potential g.
        Parameters:
        -----------
            H   :   ndarray, shape (n,m),
                    It is the matrix obtained from the difference f - C.

        Returns:
        --------
            ndarray, shape (m,),
            The log-exp regularized value of potential g.

        """
        return self._g( H - np.max( H, axis = 0 )[None,:] )  - np.max( H, axis = 0 )# Shape: (m,)

            
    def _wolfe1( self, alpha, p, slope ):
        #Armijo Condition
        """
        Here we use the Armijo condition to decide the ascent step length for updating the potentials towards the ascent direction. 
        Parameters:
        -----------
            alpha   :   float,  
                        The ascent step size.
            p   :   ndarray, shape (n,),
                    The ascent direction.
            slope   :   float,
                        It is the inner product of the gradient and p.
        Returns:
        --------
            alpha   :   float,
                        The updated ascent step size.
        """
        reduction_count = 0     
        while True:
            f_update = self.f + alpha * p# Shape: (n,)
            g_update = self._logexp_g( f_update[:,None] - self.C )# Shape: (m,)
            f_current = self.f# Shape: (n,)
            g_current = self._logexp_g( self.f[:,None] - self.C )# Shape: (m,)
            condition = self._objectivefunction( f_update, g_update ) < self._objectivefunction( f_current, g_current ) + self.c * alpha * slope
            if condition or np.isnan( self._objectivefunction( f_update, g_update ) ):
                alpha = self.rho * alpha  
                reduction_count += 1
            else:
                break
        return alpha
    
    def _precond_inversion( self, unnormalized_Hessian, gradient, rtol, atol, inversion_method, max_inversions, optType = None, print_time = False ):
        """

        Parameters:
        -----------
            unnormalized_Hessian    :   ndarray, shape: (n,n),
                                        The unnormalized Hessian.
            gradient    :   ndarray, shape: (n,),
                            The gradient of the objective function with respect to potential f.
            inversion_method    :   str,
                                    The method of inversion to be used for the Hessian. The following are the options:
                                    - "exact" : Exact inversion.
                                    - "iterative" : Iterative inversion using conjugate gradient (CG) or GMRES.
            max_inversions  :   int,
                                The number of iterative inversions to be performed to obtain the inverse of the Hessian followed by obtaining the ascent direction.
                                Defaults to -1, which indicates exact inversion.
            rtol    :   float,
                        The value of relative tolerance which is a hyperparameter to the iterative inversion algorithm, here it is conjugate gradient (CG) or GMRES.
            atol    :   float,
                        The value of absolute tolerance which is a hyperparameter to the iterative inversion algorithm, here it is conjugate gradient (CG) or GMRES.
            optType     :   str,
                            The choice of iterative inversion algorithm. The following are the options:
                            - "cg" : Conjugate Gradient.
                            - "gmres" : GMRES.
                            Defaults to None, that is, exact inversion.
            print_time  :   bool,
                            Indicator to print time stamps at different steps of preconditioned inversion of the Hessian.       


        Returns:
        --------
        Returns a tuple containing the optimal ascent direction vector p and the recorded timings of various steps of the algorithm. 
        The following are their descriptions:
            p   :   ndarray, shape: (n,),
                    The optimal ascent direction vector.
            timings     :   list,
                            The list of timestamps recorded.
        """

        def printing_time( statements ):
            """
                Function to enable the option to print time stamps at different steps.
                Parameters:
                -----------
                    statements  :   List of strings to be printed in a line.
            """
            if print_time:
                print( " ".join( statements ) )
                
        timings = []
        start = time.time()
        # Record list of unwinding transformations on final result
        unwinding_transformations = []
        # Construct modified Hessian  
        diag = 1/np.sqrt( self.a )          
        self.modified_Hessian = diag[:,None] * unnormalized_Hessian * diag[None,:]
        # Dummy variable to work on
        matrix = self.modified_Hessian
        # Preconditioning along null vector
        vector = self.null_vector# Shape: (n,)
        vector = vector/np.linalg.norm( vector )
        vector_E = vector
        # Transforming the gradient by conjugation
        gradient = diag[:,None] * gradient[:,None]
        # Unwinding transformation to obtain the original direction vector i.e., without any conjugation
        unwinding_transformations.append( lambda x : diag[:,None] * x )
        end = time.time()
        # Record timings
        interval = 1e3 * ( end - start )
        timings.append( interval )
        printing_time( [ "\n|--- Time required for initial preconditioning: ", str( np.round( interval, 5 ) ), "ms---|" ])
        # Conditioning with other vectors
        #  Naming conventions:
        #  y = Preconditioning vectors as a numpy matrix n by k
        #  matrix = our matrix A to precondition
        #  We only form the data y and z such that
        #  P = id + z*y.T
        start0 = time.time()
        y = np.array( self.precond_vectors).T # Matrix of size n by k
        # Compute eigenvalues
        Ay = np.dot( matrix, y )
        eigenvalues = np.sum( y * Ay, axis = 0 )
        # Compute data for P = id + y*diag(values)*y.T
        values = ( 1/np.sqrt(eigenvalues) - 1 )# Vector of size k
        z = y * values[None,:]
        end = time.time()
        # Record timings
        interval = 1e3 * ( end - start0 )
        timings.append( interval )
        printing_time( [ "|--- Time required for preconditioning matrix formation: ", str( np.round( interval, 5 ) ), "ms---|" ] )

        # Changing A=matrix to PAP
        start2 = time.time()
      
        def _apply_P( vector ):
            """
                Function mapping v to Pv, where P = Id + z*y.T.

                Parameters:
                ----------
                    vector  :   ndarray, shape: (n,).

                Returns:
                -------
                    ndarray, shape: (n,),  
                    vector obtained from the map Pv.

            """
            return  vector + z @ ( y.T @ vector ) 
        
   
        def _preconditioned_map( vector ):
            """
                Function mapping v to P(A+E)Pv,
                    where   P = P = Id + z*y.T,
                            A   :   ndarray, shape: (n,n),
                            vector_E    :   ndarray, shape: (n,),
                            E   :   vector_E vector_E^T.        
                Parameters:
                ----------
                    vector  :   ndarray, shape: (n,).
                
                Returns:
                -------
                    ndarray, shape: (n,),
                    The preconditioned vector. 

            """
            vector = _apply_P( vector ) 
            vector = np.dot( matrix, vector )  + vector_E * np.dot( vector_E, vector )
            vector = _apply_P( vector ) 
            return vector
        # Apply P
        # At beginning on gradient
        # At the end 
        # Preconditioning the gradient
        gradient = _apply_P( gradient )
        # The transformation to precondition the direction vector 
        unwinding_transformations.append( lambda x : _apply_P(x) )
        end = time.time()
        # Record timings
        interval = 1e3 * ( end - start2 )
        timings.append( interval )
        printing_time( [ "|--- Time required for changing A to PAP: ", str( np.round( interval, 5 ) ), "ms---|" ] )
        #
        # Solve either iteratively using CG or exactly
        start3 = time.time()
        if inversion_method == "iterative":
            self.m  = matrix
            A = scipy.sparse.linalg.LinearOperator( ( self.m.shape[0], self.m.shape[1] ), matvec = _preconditioned_map ) 
            if optType == 'cg':
                inverse, exit_code = scipy.sparse.linalg.cg(    A,
                                                                gradient, 
                                                                x0 = gradient, 
                                                                maxiter = max_inversions, 
                                                                rtol = rtol, 
                                                                atol = atol )
                # print( "  --- CG exit code: ", exit_code)
            else:
                inverse, exit_code = scipy.sparse.linalg.gmres(     A,
                                                                    gradient, 
                                                                    x0 = gradient, 
                                                                    maxiter = max_inversions, 
                                                                    rtol = rtol, 
                                                                    atol = atol )
                # print( "  --- GMRES exit code: ", exit_code)
            p_k = self.epsilon * inverse
            p_k = p_k.reshape( ( p_k.shape[0], 1 ) ) # For some reason, this outputs (n,) and the next line outputs (n,1)
        else:
          # Preconditioning along null vector                                     
          vector = vector.reshape( ( len(vector), 1 ) )
          matrix = matrix + np.dot( vector, vector.T )     
          # True Preconditioning for exact inverse 
          B = np.dot( Ay, z.T )
          C = z @ np.dot( y.T, Ay ) @ z.T
          matrix = matrix + B + B.T + C
          self.Hessian_stabilized = - matrix/self.epsilon
          p_k = - np.linalg.solve( self.Hessian_stabilized, gradient )
        end = time.time()
        # Record timings
        interval = 1e3 * ( end - start3 )
        timings.append( interval )
        printing_time( [ "|--- Time taken to invert the linear system for p_k: ", str( np.round( interval, 5 )), "ms---|" ] )
        start4 = time.time()
        # Unwind
        for transform in unwinding_transformations:
          p_k = transform( p_k )
        end = time.time()
        # Record timings
        interval = 1e3 * ( end - start4 )
        timings.append( interval )
        printing_time( [ "|--- Time taken for unwinding: ", str( np.round( interval, 5 ) ), "ms---|" ] )
        interval = 1e3 * ( end - start )
        timings.append( interval )
        printing_time( [ "|--- Time taken for the complete code block: ", str( np.round( interval, 2 ) ), "ms---|\n"] )
        return p_k.flatten(), timings
        
    def _optimize( self, tol_obj = 1e-12, tol_err = 1e-12, max_iterations = 50, inversion_method = "exact", max_inversions = 30, relative_tol = 1e-5, absolute_tol = 1e-10, optType = None, print_time = False ):
        """
        
            Parameters:
            -----------
            tol_obj :   float,
                        Tolerance to terminate the algorithm based on the comparison of the objective values using the 
                        updates of the potentials from log-domain Sinkhorn and damped Newton. 
                        Defaults to 1e-12.
            tol_err :   float,
                        Tolerance to terminate the algorithm based on the error in estimating the marginals of the coupling.
                        Defaults to 1e-12.
            max_iterations  :   int,
                                The maximum number of iterations for the optimization algorithm.
                                Defaults to 50.
            inversion_method    :   str,
                                    The method of inversion to be used for the Hessian. The following are the options:
                                    - "exact" : Exact inversion.
                                    - "iterative" : Iterative inversion using conjugate gradient (CG) or GMRES.
            max_inversions  :   int,
                                The number of iterations for the iterative inversion algorithm.
                                Defaults to 30.
            relative_tol    :   float,
                                The relative tolerance for the iterative inversion algorithm (CG or GMRES).
                                Defaults to 1e- 5.
            absolute_tol    :   float,
                                The absolute tolerance for the iterative inversion algorithm (CG or GMRES).
                                Defaults to 1e-10.
            optType     :   str,
                            The choice of iterative inversion algorithm. The following are the options:
                            - "cg" : Conjugate Gradient.
                            - "gmres" : GMRES.
                            Defaults to None, that is, exact inversion.
            print_time  :   bool,
                            Indicator to print time stamps at different steps of preconditioned inversion of the Hessian.       
            Returns:
            --------
            Returns a dictionary where the keys are strings and the values are ndarrays or list.
            The following are the keys of the dictionary and the descriptions of their values:
                potential_f     :   ndarray, shape (n,),
                                    The optimal Kantorovich potential f.
                potential_g     :   ndarray, shape (m,),
                                    The optimal Kantorovich potential g.
                errors  :   list,
                            The list of errors observed when checking conservation of mass.
                objective_values    :   list,
                                        The list of objective values observed after each ascent update.
                linesearch_steps    :   list,
                                        The list of ascent step sizes observed after each ascent update.
        """
        i = 1
        while True: 
            print( "At iteration: ", i )
            # Log-domain Sinkhorn update:
            print( "|- Log-domain Sinkhorn..." )
            start = time.time()
            ## Update f and g using log-domain Sinkhorn:
            g_log_domainSinkhorn = self._logexp_g( self.f[:,None] - self.C )# Shape: (m,)
            f_log_domainSinkhorn = self._logexp_f( g_log_domainSinkhorn[None,:] - self.C )# Shape: (n,)
            end = time.time()
            time_log_domainSinkhorn = 1e3 * ( end - start )
            print( "|-- Time taken for log-domain Sinkhorn: ",  np.round( time_log_domainSinkhorn, 5 ) ," ms." )
            ## Computing the objective value with the potentials updated using log-domain Sinkhorn
            log_domainSinkhorn_objective_value = self._objectivefunction( f_log_domainSinkhorn, g_log_domainSinkhorn )
            # Damped Newton update
            print( "|- Damped Newton..." )
            start = time.time()
            ## Computing the maximum exponent
            m_f = np.max( self.f[:, None] - self.C  , axis = 0 )# Shape: (m,)
            exp = self.a[:,None] * np.exp( ( ( self.f[:,None] - self.C - m_f[None, :] )/self.epsilon ) )# Shape: (n,m)
            ## Computing the exponentials
            sum_exp = np.sum( exp, axis = 0 )# Shape: (m,)
            ## Computing the exponentials with normalization by the sum of the exponentials along the corresponding column
            self.normalized_exp = exp/sum_exp# Shape: (n,m)
            ## Computing the gradient w.r.t f
            self.grad_f = self._computegradientf()# Shape: (n,)
            ## Computing the unnormalized Hessian
            RowSum = np.sum( self.normalized_exp * self.b[None,:], axis = 1 )# Shape: (n,)
            self.Hessian = np.diag( RowSum ) - np.dot( self.normalized_exp, np.diag( self.b ) @ self.normalized_exp.T )# Shape: (n,n)
            ## Compute solution of Ax = b:
            p_k, _ = self._precond_inversion(   self.Hessian, 
                                                self.grad_f, 
                                                inversion_method = inversion_method, 
                                                max_inversions = max_inversions,
                                                rtol = relative_tol,
                                                atol = absolute_tol,
                                                optType = optType,
                                                print_time = print_time
                                             )
            ## Computing the update step size with respect to the Armijo condition:  
            slope = np.dot( p_k, self.grad_f )
            ### Inital step size
            alpha = 1
            ### Updated step size
            alpha = self._wolfe1( alpha, p_k, slope )
            self.alpha_list.append( alpha )
            ## Update f and g using damped Newton:
            f_damped_Newton = self.f + alpha * p_k# Shape: (n,)
            g_damped_Newton = self._logexp_g( f_damped_Newton[:,None] - self.C )# Shape: (m,)
            end = time.time()
            time_dampedNewton = 1e3 * ( end - start )
            print( "|-- Time taken for damped Newton: ",  np.round( time_dampedNewton, 5 ) ," ms." )
            # Computing the objective value with the potentials updated using damped Newton
            damped_Newton_objective_value = self._objectivefunction( f_damped_Newton, g_damped_Newton )
            ## Compare the objective function value for the two updates and choosing the one with greater increment
            if log_domainSinkhorn_objective_value > damped_Newton_objective_value:
                print( "|- Updating using log-domain Sinkhorn." )
                self.f = f_log_domainSinkhorn# Shape: (n,)
                self.g = g_log_domainSinkhorn# Shape: (m,)
                self.update_indicator[ i - 1 ] = "log-domain Sinkhorn" 
            else:
                print( "|- Updating using damped Newton." )
                self.f = f_damped_Newton# Shape: (n,)
                self.g = g_damped_Newton# Shape: (m,)
                self.update_indicator[ i - 1 ] = "Damped Newton"
            # Computing the coupling matrix
            P = self.a[:,None] * np.exp( ( self.f[:,None] + self.g[None,:]  - self.C )/self.epsilon ) * self.b[None,:]# Shape: (n,m), line (*)
            # Check conservation of mass: ||P1_m - a||_1 + ||P^T 1_n - b||_1
            self.errors.append( np.linalg.norm( np.sum( P, axis = 1 ) - self.a, ord = 1 )
                                +
                                np.linalg.norm( np.sum( P, axis = 0 ) - self.b, ord = 1 )
                              )
            # Evaluating objective function after the ascent update
            value = self._objectivefunction( self.f, self.g )
            self.objective_values.append( value ) 
            # Condition to terminate the algorithm based on the objective values using the potentials from updated using the log-domain Sinkhorn and damped Newton
            condition1 = abs( log_domainSinkhorn_objective_value - damped_Newton_objective_value ) > tol_obj
            # Condition to terminate the algorithm based on the estimation error of the marginals from the coupling
            condition2 = self.errors[-1] > tol_err
            # Combining the two conditions
            condition = condition1 or condition2
            if i + 1 <= max_iterations and condition:
                i = i + 1
            else:
                break
        # Change of convention because of line (*)
        self.f = self.f + self.epsilon * np.log( self.a )
        self.g = self.g + self.epsilon * np.log( self.b )
        # end for           
        return {
            "potential_f"       : self.f,
            "potential_g"       : self.g,
            "errors"            : self.errors,
            "objective_values"  : self.objective_values,
            "linesearch_steps"  : self.alpha_list,
            "update_indicator"  : self.update_indicator,
        }


# Experiments

#### Helper functions

In [ ]:
"""To compute distance matrix"""
def distmat( x, y ):
    return np.sum( x ** 2, 0 )[:,None] + np.sum( y ** 2, 0 )[None,:] - 2 * x.transpose().dot( y )

"""To Normalise a vector"""
normalize = lambda a: a/np.sum( a )

"""To Compute P"""
def GetP( u, K, v ):
    return u[:,None] * K * v[None,:]

In [ ]:
def generate_data( N ):
    """
     N is a list of the size of the data on x and y
    """
    # Square
    x = np.random.rand( 2, N[0] ) - 0.5
    # Annulus
    theta = 2 * np.pi * np.random.rand( 1, N[1] )  
    r = 0.8 + .2 * np.random.rand( 1, N[1] )
    y = np.vstack( ( r * np.cos( theta ), r * np.sin( theta ) ) )
    return x, y

#### Sampling point clouds

In [ ]:
N = [ 1100, 1000 ]
# N = [ 600, 500 ]
x, y = generate_data( N )                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   
#Cost matrix
C = distmat( x, y )
# a and b
a = normalize( np.ones( N[0] ) )
b = normalize( np.ones( N[1] ) )

## I. Sinkhorn: Highlighting failure of the algorithm for small $\varepsilon$

Here we can observe that from $\varepsilon \leq 0.0009$ the algorithm terminates because of underflow and overflow of values.

In [ ]:
sinkhorn_epsilons = [ 1.0, 0.5, 0.05, 0.005, 0.001, 0.0009, 0.0007, 0.0005, 0.0003, 0.0001 ]

In [ ]:
# Sinkhorn
print("Sinkhorn.... ")
print( "Doing for (",N[0], N[1],")." )
SinkhornP = []
results_Sinkhorn = []
times_Sinkhorn = []
for epsilon in sinkhorn_epsilons:
  print( "For epsilon = "+str(epsilon)+":" )    
  # Kernel
  K = np.exp( - C/epsilon )
  print( " |- Iterating" )
  u = a
  v = b
  optimizer = sinkhorn( K, a, b, u, v, epsilon )
  start = time.time()
  out = optimizer._optimize( max_iterations = 10000 )
  end = time.time()
  results_Sinkhorn.append( out )
  times_Sinkhorn.append( end - start )
  print( " |- Computing P" )
  print( "" )
  u_opt = np.exp( out['potential_f']/epsilon )
  K = np.exp( - C/epsilon )
  v_opt =  np.exp( out['potential_g']/epsilon )
  P_opt = GetP( u_opt, K, v_opt )
  SinkhornP.append( P_opt )
# end for

### Error plot

In [ ]:
plt.rcParams.update( { 'font.size' : 12 } )
plt.figure( figsize = ( 20, 5 ) )
plt.title( "$||P1 -a||_1+||P1 -b||_1$" )
for i in range( len(results_Sinkhorn) ):
  errors = np.asarray( results_Sinkhorn[i]['errors'] )
  plt.plot( errors, label = 'Sinkhorn for $\epsilon = $'+ str(sinkhorn_epsilons[i]), linewidth = 2 )
# end for
plt.yscale( 'log' )
plt.legend( loc = "upper right" )
plt.savefig( image_folder_path + "/Error_Sinkhorn.pdf", format = 'pdf' )
plt.show()

## II. Log-domain Sinkhorn: For obtaining the preconditioning vectors

In [ ]:
log_domainSinkhorn_epsilons = [ 1.0, 0.5, 0.1, 0.05, 0.01, 0.005, 0.001 ]
log_domainSinkhorn_epsilons = [ 1.0, 0.5, 0.1, 0.05, 0.01 ]

In [ ]:
# Log domain Sinkhorn
print( "Log domain Sinkhorn... " )
print( "Doing for (",N[0],N[1],")." )
a = normalize( np.ones( N[0] ) )
b = normalize( np.ones( N[1] ) )
results_log_domainSinkhorn = []
times_log_domainSinkhorn   = []
log_domainSinkhornP        = []
# Cost matrix
C = distmat( x, y )
for epsilon in log_domainSinkhorn_epsilons:
  print( "For epsilon = "+str(epsilon)+":" )    
  print( " |- Iterating" )
  optimizer = log_domainSinkhorn( a, b, C, epsilon )
  start = time.time()
  out = optimizer._optimize( max_iterations = 20000 )
  end = time.time()
  print( "Terminating after iteration: ", out['iterations'] )
  results_log_domainSinkhorn.append( out )
  times_log_domainSinkhorn.append( end - start )
  print( " |- Computing P" )
  print( "" )
  u_opt = np.exp( out['potential_f']/epsilon )
  K = np.exp( - C/epsilon )
  v_opt =  np.exp( out['potential_g']/epsilon )
  P_opt = GetP( u_opt, K, v_opt )
  log_domainSinkhornP.append( P_opt )
# end for

### Error plot

In [ ]:
plt.rcParams.update( { 'font.size' : 12 } )
plt.figure( figsize = ( 20, 5 ) )
plt.title( "$||P1 -a||_1+||P1 -b||_1$" )
for i in range( len( results_log_domainSinkhorn) ):
  errors = np.asarray( results_log_domainSinkhorn[i]['errors'] )
  plt.plot( errors, label = 'Log-sinkhorn for $\epsilon = $'+ str(log_domainSinkhorn_epsilons[i]), linewidth = 2 )
# end for
plt.yscale( 'log' )
plt.legend( loc = "upper right" )                               
plt.xlabel( "Iterations" )
plt.ylabel( "Error in log-scale" )
plt.savefig( image_folder_path + "/Error_log_exp_Sinkhorn.pdf", format = 'pdf' )
plt.show()

### Forming the Hessians for different values of $\varepsilon$

In [ ]:
log_domainSinkhorn_Hessians = {}
for i in range( len( log_domainSinkhorn_epsilons ) ):
    f = results_log_domainSinkhorn[i]["potential_f"]
    # Computing the maximum exponent
    m_f = np.max( f[:, None] - C  , axis = 0 )# Shape: (m,)
    exp = a[:,None] * np.exp( ( ( f[:,None] - C - m_f[None, :] )/log_domainSinkhorn_epsilons[i] ) )# Shape: (n,m)
    # Computing the exponentials
    sum_exp = np.sum( exp, axis = 0 )# Shape: (m,)
    # Computing the exponentials with normalization by the sum of the exponentials along the corresponding column
    normalized_exp = exp/sum_exp# Shape: (n,m)
    # Computing the unnormalized Hessian
    RowSum = np.sum( normalized_exp * b[None,:], axis = 1 )# Shape: (n,)
    Hessian = np.diag( RowSum ) - np.dot( normalized_exp, np.diag( b ) @ normalized_exp.T )# Shape: (n,n)
    diag = 1/np.sqrt( a )
    log_domainSinkhorn_Hessians[ log_domainSinkhorn_epsilons[i] ] =  diag[:,None] * Hessian * diag[None,:]

### Spectral statistics

In [ ]:
def spectral_decomposition( mat ):
    eig, v = np.linalg.eigh( mat )
    sorting_indices = np.argsort( eig )
    eig = eig[ sorting_indices ]
    v   = v[ : , sorting_indices ]
    print( "List of smallest eigenvalues: ", eig[ : 10 ] )
    print( "List of largest  eigenvalues: ", eig[ - 10 : ] )
    return eig, v

In [ ]:
eigs = []
eigvecs = []
for i in range( len( log_domainSinkhorn_epsilons ) ):
    eps = log_domainSinkhorn_epsilons[i]
    print( "Spectral statistics of Hessian for epsilon = "+str(eps) )
    result = log_domainSinkhorn_Hessians[ log_domainSinkhorn_epsilons[i] ] 
    ev = spectral_decomposition( result )
    eigs.append( ev[0] )    
    eigvecs.append( ev[1] )
    print( "" )  
# end for

In [ ]:
plt.rcParams.update( { 'font.size' : 10 } )
fig, ax = plt.subplots( figsize = ( 5, 12 ), nrows =  len( log_domainSinkhorn_epsilons ), ncols = 1, sharey = True )
plt.title( " Histogram of eigenvalues. " )
for i in range( len( log_domainSinkhorn_epsilons ) ):
    ax[i].hist( eigs[i], 50 )
    ax[i].set_title( " $\epsilon$: " + str( log_domainSinkhorn_epsilons[i] ) )
    ax[i].set_xlabel( " Eigenvalues " )
    ax[i].set_yscale( "log" )
    ax[i].set_xlim( 0, 1 )
# end for
plt.subplots_adjust( wspace = 0, hspace = 0.5 ) 
plt.tight_layout()
plt.savefig( image_folder_path + "/Spectral_plot.pdf", format = 'pdf' )
plt.show()

In [ ]:
def build_preconditioners( num_eigs, modified_Hessian, ansatz = True ):
    # Diagonalize
    eigenvalues, eigenvectors = np.linalg.eigh( modified_Hessian )
    sorting_indices = np.argsort( eigenvalues  )
    eigenvalues  = eigenvalues[ sorting_indices ]
    eigenvectors = eigenvectors[ : , sorting_indices ]
    # Form null vector
    if not ansatz:
        null_vector = eigenvectors[:, 0]
    else:
        null_vector = np.ones( N[0] ) 
        norm = np.sqrt( N[0] )
        null_vector = null_vector/norm
    # Form other vectors
    indices = []
    for i in range( num_eigs ):
        indices.append( i + 1 )
    # end for
    precond_vectors = []
    for index in indices:
        precond_vectors.append( eigenvectors[ :, index ] )
    # end for
    return null_vector, precond_vectors

### Effect of preconditioning on spectrum for various number of preconditioning eigenvectors

In [ ]:
num_eigs = [ 0, 10, 20, 30, 40, 50, 100 ]
preconditioned_Hessians = {}
for numeigs  in  range( len( num_eigs ) ):
    preconditioned_Hessians[ num_eigs[ numeigs ] ] = []
    for i in  range( len( log_domainSinkhorn_epsilons ) ):
        result =  log_domainSinkhorn_Hessians[ log_domainSinkhorn_epsilons[i] ]
        if num_eigs[ numeigs ] != 0:
            null_vector, precond_vectors = build_preconditioners( num_eigs[ numeigs ], result, ansatz = False )
            y_ = np.array( precond_vectors ).T# Matrix of size n by k
            # Compute eigenvalues
            Ay = np.dot( result, y_ )
            eigenvalues = np.sum( y_ * Ay, axis = 0 )
            # Compute P_matrix = id + y*diag(values)*y.T
            values = ( 1/np.sqrt(eigenvalues) - 1 )# Vector of size k
            z = y_ * values[None,:]
            B = np.dot( Ay, z.T )
            C_ = z @ np.dot( y_.T, Ay ) @ z.T
            result = result + B + B.T + C_
        preconditioned_Hessians[ num_eigs[ numeigs ] ].append( result )
    # end for
# end for

In [ ]:
eigs = {}
for numeigs in  range( len( num_eigs ) ):
    eigs[ num_eigs[ numeigs ] ] = []
    for i in range( len( log_domainSinkhorn_epsilons ) ):
        eps = log_domainSinkhorn_epsilons[i]
        print( "Spectral statistics of Hessian for epsilon = "+str(eps) )
        ev = spectral_decomposition( preconditioned_Hessians[ num_eigs[ numeigs ] ][ i ] )
        eigs[ num_eigs[ numeigs ] ].append( ev[0] )
        print("")
    # end for
# end for

#### Sprectral plots showing the change in the spectrum of Hessian after preconditioning with different number of preconditioning vectors

In [ ]:
plt.rcParams.update( { 'font.size' : 30 } )
fig, ax = plt.subplots( figsize = ( 50, 60 ), nrows = len(num_eigs), ncols = len(log_domainSinkhorn_epsilons), sharey = True, sharex = False )
p = np.log10( 0.5 )   
for numeigs in range( len( num_eigs ) ):
    for i in range( len( log_domainSinkhorn_epsilons ) ):
        ax[ numeigs ][i].hist( eigs[ num_eigs[ numeigs ] ][i], 50, rwidth = 0.9 )
        ax[ numeigs ][i].set_title( " k = "+str(num_eigs[ numeigs ])+", $\epsilon$ = " +str(log_domainSinkhorn_epsilons[i])+ "" )
        ax[ numeigs ][i].set_ylim( ymin = 10**p )
        ax[ numeigs ][i].set_yscale( "log" )    
    # end for
# end for
ax[ len(num_eigs) - 1 ][ len( log_domainSinkhorn_epsilons ) - 1 ].set_xticks( [ 0, 1 ] )  
plt.subplots_adjust( wspace = 0.1, hspace = 0.2 )
plt.savefig( image_folder_path + "/Effect_of_preconditioning.pdf", format = 'pdf' )
plt.show()

## III. Iterative optimization

### i. Preconditioning $\varepsilon = 0.5,\ \rho = 0.3,\ c = 0.05$

#### Using exact method for Hessian inversion

In [ ]:
epsilons_iterative_optim = [ 0.05, 0.01, 0.005, 0.001, 0.0009, 0.0007, 0.0005, 0.0003, 0.0001 ]
# Number of preconditioning eigenvectors, adjusted to be within the bounds of the array dimensions
num_eigs = 35
# Choosing the epsilon corresponding to which the Hessian is used to obtain the preconditioning vectors
preconditioning_epsilon = 0.5
null_vector, precond_vectors = build_preconditioners( num_eigs, log_domainSinkhorn_Hessians[ preconditioning_epsilon ], ansatz = False )

In [ ]:
print( " Doing for (",N[0], N[1],"). " )
# Damping factor for ascent step-size
rho = 0.3
# Sufficient increase parameter in the Armijo condition 
c = 0.05
# Number of iterations
num_iterations = 50
# Inversion method
method = "exact"
out_exact = {}
timings_exact = {}
f = None
for epsilon in epsilons_iterative_optim :
    print( "\n For epsilon = "+str(epsilon)+":\n" )    
    # Initializing potential f
    if f is None:
        f = a * 0  
    print( " Iterating" )
    optimizer =  iterative_optimization(    C,
                                            a,
                                            b,
                                            f,
                                            epsilon,
                                            rho,
                                            c,
                                            null_vector,
                                            precond_vectors[:]
                                        )    
    start = time.time()                                                  
    out_exact[epsilon] = optimizer._optimize(   max_iterations = num_iterations,
                                                inversion_method = method,
                                                optType = None,
                                                print_time = False
                                            )
    end = time.time() 
    # Recording time taken for each epsilon
    timings_exact[epsilon] = end - start
# end for 

##### Error plot

In [ ]:
plt.rcParams.update( { 'font.size' : 12 } )
plt.figure( figsize = ( 20, 7 ) )   
plt.title( "$||P1 -a||_1+||P1 -b||_1$" ) 
for epsilon in out_exact.keys():
    colors  =   []
    errors  =   []
    for i in range( len( out_exact[epsilon]['errors'] ) ):
        errors.append( out_exact[epsilon]['errors'][i] )
        if out_exact[epsilon]['update_indicator'][i] == 'log-domain Sinkhorn':
            colors.append( 'red' )# If log-domain Sinkhorn update was used at this step
        else:
            colors.append( "lightgreen" )# If damped Newton was used at this step
    # end for
    plt.plot( errors,  marker = 'o', label = 'For $\epsilon = $'+str(epsilon) )
    # Plot each marker with a different color
    for i in range( len( colors ) ):
        plt.scatter( i, errors[i], color = colors[i], s = 50, edgecolors = 'black', zorder = 2 )
    # end for
# end for
# Add legend entries for the markers (plot dummy points for legend)
plt.scatter( [], [], color = 'red', s = 50, edgecolors = 'black', label = 'Log-domain Sinkhorn' )# Dummy red marker
plt.scatter( [], [], color = "lightgreen", s = 50, edgecolors = 'black', label = 'Semi-dual damped Newton' )# Dummy blue marker
plt.xlabel( " Number of iterations " )  
plt.ylabel( " Error in log-scale " )  
plt.legend( loc = "upper right" ) 
plt.yscale( 'log' ) 
plt.savefig( image_folder_path + "/Error_iterative_optimization_exact_inversion1.pdf", format = 'pdf' )
plt.show() 


##### Objective function

In [ ]:
plt.rcParams.update( { 'font.size' : 12 } )
plt.figure( figsize = ( 20, 7 ) )   
plt.title( "Objective function:  < f, a > + < g, b > " )
for epsilon in out_exact.keys():
    colors = []
    values = []
    for i in range( len( out_exact[epsilon]['objective_values'] ) ):
        try:
            values.append( out_exact[epsilon]['objective_values'][i] )
            if out_exact[epsilon]['update_indicator'][i] == 'log-domain Sinkhorn':
                colors.append( 'red' )# If log-domain Sinkhorn update was used at this step
            else:
                colors.append( "lightgreen" )# If damped Newton was used at this step
        except:
            break
    # end for
    plt.plot( values,  marker = 'o', label = 'For $\epsilon = $'+str(epsilon) )
    # Plot each marker with a different color
    for i in range( len( colors ) ):
        plt.scatter( i, values[i], color = colors[i], s = 50, edgecolors = 'black', zorder = 2 )
    # end for
# end for
# Add legend entries for the markers (plot dummy points for legend)
plt.scatter( [], [], color = 'red', s = 50, edgecolors = 'black', label = 'Log-domain Sinkhorn' )# Dummy red marker
plt.scatter( [], [], color = "lightgreen", s = 50, edgecolors = 'black', label = 'Semi-dual damped Newton' )# Dummy blue marker
plt.xlabel( " Number of iterations " )
plt.ylabel( " Objective value " )
plt.legend( loc = "upper right" )
plt.savefig( image_folder_path + "/Objective_function_iterative_optimization_exact_inversion1.pdf", format = 'pdf' )
plt.show()

#### Using iterative method for Hessian inversion

In [ ]:
epsilons_iterative_optim = [ 0.05, 0.01, 0.005, 0.001, 0.0009, 0.0007, 0.0005, 0.0003, 0.0001 ]
# Number of preconditioning eigenvectors, adjusted to be within the bounds of the array dimensions
num_eigs = 35
# Choosing the epsilon corresponding to which the Hessian is used to obtain the preconditioning vectors
preconditioning_epsilon = 0.5
null_vector, precond_vectors = build_preconditioners( num_eigs, log_domainSinkhorn_Hessians[ preconditioning_epsilon ], ansatz = False )

In [ ]:
print( " Doing for (",N[0], N[1],"). " )
# Damping factor for ascent step-size
rho = 0.3
# Sufficient increase parameter in the Armijo condition 
c = 0.05
# Absolute tolerance parameeter in iterative inversion
a_tol = 1e-12
# Relative tolerance parameter in iterative inversion
r_tol = 1e-5
# Number of iterations
num_iterations = 50
# Maximum number of iterative iversions
max_inv = 30
# Inversion method
method = "iterative"
# Iterative inversion method
iterative_solver = 'cg'
timings_iterative = {}
out_iterative = {}
f = None
for epsilon in epsilons_iterative_optim :
    print( "\n For epsilon = "+str(epsilon)+":\n" )    
     # Initializing potential f
    if f is None:
        f   =   a * 0 
    print( " Iterating" )
    optimizer =  iterative_optimization(    C,
                                            a,
                                            b,
                                            f,
                                            epsilon,
                                            rho,
                                            c,
                                            null_vector,
                                            precond_vectors[:]
                                        )
    start = time.time()                                                      
    out_iterative[epsilon] = optimizer._optimize(   max_iterations = num_iterations,
                                                    inversion_method = method,
                                                    max_inversions = max_inv,
                                                    relative_tol = r_tol,
                                                    absolute_tol = a_tol,
                                                    optType = iterative_solver,
                                                    print_time = False
                                                )
    end = time.time() 
    # Recording time taken for each epsilon
    timings_iterative[epsilon] = end - start
# end for              

##### Error plot

In [ ]:
plt.rcParams.update( { 'font.size' : 12 } )
plt.figure( figsize = ( 20, 7 ) )   
plt.title( "$||P1 -a||_1+||P1 -b||_1$" ) 
for epsilon in out_iterative.keys():
    colors  =   []
    errors  =   []
    for i in range( len( out_iterative[epsilon]['errors'] ) ):
        errors.append( out_iterative[epsilon]['errors'][i] )
        if out_iterative[epsilon]['update_indicator'][i] == 'log-domain Sinkhorn':
            colors.append( 'red' )# If log-domain Sinkhorn update was used at this step
        else:
            colors.append( "lightgreen" )# If damped Newton was used at this step
    # end for
    plt.plot( errors,  marker = 'o', label = 'For $\epsilon = $'+str(epsilon) )
    # Plot each marker with a different color
    for i in range( len( colors ) ):
        plt.scatter( i, errors[i], color = colors[i], s = 50, edgecolors = 'black', zorder = 2 )
    # end for
# end for
# Add legend entries for the markers (plot dummy points for legend)
plt.scatter( [], [], color = 'red', s = 50, edgecolors = 'black', label = 'Log-domain Sinkhorn' )# Dummy red marker
plt.scatter( [], [], color = "lightgreen", s = 50, edgecolors = 'black', label = 'Semi-dual damped Newton' )# Dummy blue marker
plt.xlabel( " Number of iterations " )  
plt.ylabel(  " Error in log-scale " )  
plt.legend( loc = "upper right" ) 
plt.yscale( 'log' ) 
plt.savefig( image_folder_path + "/Error_iterative_optimization_iterative_inversion1.pdf", format = 'pdf' )
plt.show() 

##### Objective function

In [ ]:
plt.rcParams.update( { 'font.size' : 12 } )
plt.figure( figsize = ( 20, 7 ) )   
plt.title( "Objective function:  < f, a > + < g, b > " )
for epsilon in out_iterative.keys():
    colors = []
    values = []
    for i in range( len( out_iterative[epsilon]['objective_values'] ) ):
        try:
            values.append( out_iterative[epsilon]['objective_values'][i] )
            if out_iterative[epsilon]['update_indicator'][i] == 'log-domain Sinkhorn':
                colors.append( 'red' )# If log-domain Sinkhorn update was used at this step
            else:
                colors.append( "lightgreen" )# If damped Newton was used at this step
        except:
            break
    # end for
    plt.plot( values,  marker = 'o', label = 'For $\epsilon = $'+str(epsilon) )
    # Plot each marker with a different color
    for i in range( len( colors ) ):
        plt.scatter( i, values[i], color = colors[i], s = 50, edgecolors = 'black', zorder = 2 )
    # end for
# end for
# Add legend entries for the markers (plot dummy points for legend)
plt.scatter( [], [], color = 'red', s = 50, edgecolors = 'black', label = 'Log-domain Sinkhorn' )# Dummy red marker
plt.scatter( [], [], color = "lightgreen", s = 50, edgecolors = 'black', label = 'Semi-dual damped Newton' )# Dummy blue marker
plt.xlabel( " Number of iterations " )
plt.ylabel( " Objective value " )
plt.legend( loc = "upper right" )
plt.savefig( image_folder_path + "/Objective_function_iterative_optimization_iterative_inversion1.pdf", format = 'pdf' )
plt.show()

#### Time comparison between exact and iterative inversion

In [ ]:
plt.rcParams.update( { 'font.size' : 12 } )                                                                                                                          
plt.figure( figsize = ( 20, 7 ) )   
plt.title( "$$" ) 
plt.title( "Time plot" ) 
plt.plot( list( timings_exact.values() )[::-1], label = "Iterative optimization with exact method for Hessian inversion",  linewidth = 2, marker = 'o' ) 
plt.plot( list( timings_iterative.values() )[::-1], label = "Iterative optimization with iterative method for Hessian inversion",  linewidth = 2, marker = 'o' ) 
plt.xticks( np.arange( len( epsilons_iterative_optim ) ), epsilons_iterative_optim[::-1] ) 
plt.xlabel( "$\epsilon$" )  
plt.ylabel( "Time in sec" ) 
plt.legend( loc = "upper right" )
plt.yscale( 'log' )                                                                                                                                                                                                                                                               
plt.savefig( image_folder_path + "/Timeplot_exact_vs_iterative_inversion1.pdf", format = 'pdf' ) 
plt.show()                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        

### ii. Preconditioning $\varepsilon = 0.1,\ \rho = 0.3,\ c = 0.01$

#### Using exact method for Hessian inversion

In [ ]:
epsilons_iterative_optim = [ 0.05, 0.01, 0.005, 0.001, 0.0009, 0.0007, 0.0005, 0.0003, 0.0001 ]
# Number of preconditioning eigenvectors, adjusted to be within the bounds of the array dimensions
num_eigs = 35
# Choosing the epsilon corresponding to which the Hessian is used to obtain the preconditioning vectors
preconditioning_epsilon = 0.1
null_vector, precond_vectors = build_preconditioners( num_eigs, log_domainSinkhorn_Hessians[ preconditioning_epsilon ], ansatz = False )

In [ ]:
print( " Doing for (",N[0], N[1],"). " )
# Damping factor for ascent step-size
rho = 0.3
# Sufficient increase parameter in the Armijo condition 
c = 0.01
# Number of iterations
num_iterations = 50
# Inversion method
method = "exact"
out_exact = {}
timings_exact = {}
f = None
for epsilon in epsilons_iterative_optim :
    print( "\n For epsilon = "+str(epsilon)+":\n" )    
    # Initializing potential f
    if f is None:
        f = a * 0  
    print( " Iterating" )
    optimizer =  iterative_optimization(    C,
                                            a,
                                            b,
                                            f,
                                            epsilon,
                                            rho,
                                            c,
                                            null_vector,
                                            precond_vectors[:]
                                        )    
    start = time.time()                                                  
    out_exact[epsilon] = optimizer._optimize(   max_iterations = num_iterations,
                                                inversion_method = method,
                                                optType = None,
                                                print_time = False
                                            )
    end = time.time() 
    # Recording time taken for each epsilon
    timings_exact[epsilon] = end - start
# end for 

##### Error plot

In [ ]:
plt.rcParams.update( { 'font.size' : 12 } )
plt.figure( figsize = ( 20, 7 ) )   
plt.title( "$||P1 -a||_1+||P1 -b||_1$" ) 
for epsilon in out_exact.keys():
    colors  =   []
    errors  =   []
    for i in range( len( out_exact[epsilon]['errors'] ) ):
        errors.append( out_exact[epsilon]['errors'][i] )
        if out_exact[epsilon]['update_indicator'][i] == 'log-domain Sinkhorn':
            colors.append( 'red' )# If log-domain Sinkhorn update was used at this step
        else:
            colors.append( "lightgreen" )# If damped Newton was used at this step
    # end for
    plt.plot( errors,  marker = 'o', label = 'For $\epsilon = $'+str(epsilon) )
    # Plot each marker with a different color
    for i in range( len( colors ) ):
        plt.scatter( i, errors[i], color = colors[i], s = 50, edgecolors = 'black', zorder = 2 )
    # end for
# end for
# Add legend entries for the markers (plot dummy points for legend)
plt.scatter( [], [], color = 'red', s = 50, edgecolors = 'black', label = 'Log-domain Sinkhorn' )# Dummy red marker
plt.scatter( [], [], color = "lightgreen", s = 50, edgecolors = 'black', label = 'Semi-dual damped Newton' )# Dummy blue marker
plt.xlabel( " Number of iterations " )  
plt.ylabel( " Error in log-scale " )  
plt.legend( loc = "upper right" ) 
plt.yscale( 'log' ) 
plt.savefig( image_folder_path + "/Error_iterative_optimization_exact_inversion2.pdf", format = 'pdf' )
plt.show() 


##### Objective function

In [ ]:
plt.rcParams.update( { 'font.size' : 12 } )
plt.figure( figsize = ( 20, 7 ) )   
plt.title( "Objective function:  < f, a > + < g, b > " )
for epsilon in out_exact.keys():
    colors = []
    values = []
    for i in range( len( out_exact[epsilon]['objective_values'] ) ):
        try:
            values.append( out_exact[epsilon]['objective_values'][i] )
            if out_exact[epsilon]['update_indicator'][i] == 'log-domain Sinkhorn':
                colors.append( 'red' )# If log-domain Sinkhorn update was used at this step
            else:
                colors.append( "lightgreen" )# If damped Newton was used at this step
        except:
            break
    # end for
    plt.plot( values,  marker = 'o', label = 'For $\epsilon = $'+str(epsilon) )
    # Plot each marker with a different color
    for i in range( len( colors ) ):
        plt.scatter( i, values[i], color = colors[i], s = 50, edgecolors = 'black', zorder = 2 )
    # end for
# end for
# Add legend entries for the markers (plot dummy points for legend)
plt.scatter( [], [], color = 'red', s = 50, edgecolors = 'black', label = 'Log-domain Sinkhorn' )# Dummy red marker
plt.scatter( [], [], color = "lightgreen", s = 50, edgecolors = 'black', label = 'Semi-dual damped Newton' )# Dummy blue marker
plt.xlabel( " Number of iterations " )
plt.ylabel( " Objective value " )
plt.legend( loc = "upper right" )
plt.savefig( image_folder_path + "/Objective_function_iterative_optimization_exact_inversion2.pdf", format = 'pdf' )
plt.show()

#### Using iterative method for Hessian inversion

In [ ]:
epsilons_iterative_optim = [ 0.05, 0.01, 0.005, 0.001, 0.0009, 0.0007, 0.0005, 0.0003, 0.0001 ]
# Number of preconditioning eigenvectors, adjusted to be within the bounds of the array dimensions
num_eigs = 35
# Choosing the epsilon corresponding to which the Hessian is used to obtain the preconditioning vectors
preconditioning_epsilon = 0.1
null_vector, precond_vectors = build_preconditioners( num_eigs, log_domainSinkhorn_Hessians[ preconditioning_epsilon ], ansatz = False )

In [ ]:
print( " Doing for (",N[0], N[1],"). " )
# Damping factor for ascent step-size
rho = 0.3
# Sufficient increase parameter in the Armijo condition 
c = 0.01
# Absolute tolerance parameeter in iterative inversion
a_tol = 1e-12
# Relative tolerance parameter in iterative inversion
r_tol = 1e-5
# Number of iterations
num_iterations = 50
# Maximum number of iterative iversions
max_inv = 30
# Inversion method
method = "iterative"
# Iterative inversion method
iterative_solver = 'cg'
timings_iterative = {}
out_iterative = {}
f = None
for epsilon in epsilons_iterative_optim :
    print( "\n For epsilon = "+str(epsilon)+":\n" )    
     # Initializing potential f
    if f is None:
        f   =   a * 0 
    print( " Iterating" )
    optimizer =  iterative_optimization(    C,
                                            a,
                                            b,
                                            f,
                                            epsilon,
                                            rho,
                                            c,
                                            null_vector,
                                            precond_vectors[:]
                                        )
    start = time.time()                                                      
    out_iterative[epsilon] = optimizer._optimize(   max_iterations = num_iterations,
                                                    inversion_method = method,
                                                    max_inversions = max_inv,
                                                    relative_tol = r_tol,
                                                    absolute_tol = a_tol,
                                                    optType = iterative_solver,
                                                    print_time = False
                                                )
    end = time.time() 
    # Recording time taken for each epsilon
    timings_iterative[epsilon] = end - start
# end for              

##### Error plot

In [ ]:
plt.rcParams.update( { 'font.size' : 12 } )
plt.figure( figsize = ( 20, 7 ) )   
plt.title( "$||P1 -a||_1+||P1 -b||_1$" ) 
for epsilon in out_iterative.keys():
    colors  =   []
    errors  =   []
    for i in range( len( out_iterative[epsilon]['errors'] ) ):
        errors.append( out_iterative[epsilon]['errors'][i] )
        if out_iterative[epsilon]['update_indicator'][i] == 'log-domain Sinkhorn':
            colors.append( 'red' )# If log-domain Sinkhorn update was used at this step
        else:
            colors.append( "lightgreen" )# If damped Newton was used at this step
    # end for
    plt.plot( errors,  marker = 'o', label = 'For $\epsilon = $'+str(epsilon) )
    # Plot each marker with a different color
    for i in range( len( colors ) ):
        plt.scatter( i, errors[i], color = colors[i], s = 50, edgecolors = 'black', zorder = 2 )
    # end for
# end for
# Add legend entries for the markers (plot dummy points for legend)
plt.scatter( [], [], color = 'red', s = 50, edgecolors = 'black', label = 'Log-domain Sinkhorn' )# Dummy red marker
plt.scatter( [], [], color = "lightgreen", s = 50, edgecolors = 'black', label = 'Semi-dual damped Newton' )# Dummy blue marker
plt.xlabel( " Number of iterations " )  
plt.ylabel(  " Error in log-scale " )  
plt.legend( loc = "upper right" ) 
plt.yscale( 'log' ) 
plt.savefig( image_folder_path + "/Error_iterative_optimization_iterative_inversion2.pdf", format = 'pdf' )
plt.show() 

##### Objective function

In [ ]:
plt.rcParams.update( { 'font.size' : 12 } )
plt.figure( figsize = ( 20, 7 ) )   
plt.title( "Objective function:  < f, a > + < g, b > " )
for epsilon in out_iterative.keys():
    colors = []
    values = []
    for i in range( len( out_iterative[epsilon]['objective_values'] ) ):
        try:
            values.append( out_iterative[epsilon]['objective_values'][i] )
            if out_iterative[epsilon]['update_indicator'][i] == 'log-domain Sinkhorn':
                colors.append( 'red' )# If log-domain Sinkhorn update was used at this step
            else:
                colors.append( "lightgreen" )# If damped Newton was used at this step
        except:
            break
    # end for
    plt.plot( values,  marker = 'o', label = 'For $\epsilon = $'+str(epsilon) )
    # Plot each marker with a different color
    for i in range( len( colors ) ):
        plt.scatter( i, values[i], color = colors[i], s = 50, edgecolors = 'black', zorder = 2 )
    # end for
# end for
# Add legend entries for the markers (plot dummy points for legend)
plt.scatter( [], [], color = 'red', s = 50, edgecolors = 'black', label = 'Log-domain Sinkhorn' )# Dummy red marker
plt.scatter( [], [], color = "lightgreen", s = 50, edgecolors = 'black', label = 'Semi-dual damped Newton' )# Dummy blue marker
plt.xlabel( " Number of iterations " )
plt.ylabel( " Objective value " )
plt.legend( loc = "upper right" )
plt.savefig( image_folder_path + "/Objective_function_iterative_optimization_iterative_inversion2.pdf", format = 'pdf' )
plt.show()

#### Time comparison between exact and iterative inversion

In [ ]:
plt.rcParams.update( { 'font.size' : 12 } )                                                                                                                          
plt.figure( figsize = ( 20, 7 ) )   
plt.title( "$$" ) 
plt.title( "Time plot" ) 
plt.plot( list( timings_exact.values() )[::-1], label = "Iterative optimization with exact method for Hessian inversion",  linewidth = 2, marker = 'o' ) 
plt.plot( list( timings_iterative.values() )[::-1], label = "Iterative optimization with iterative method for Hessian inversion",  linewidth = 2, marker = 'o' ) 
plt.xticks( np.arange( len( epsilons_iterative_optim ) ), epsilons_iterative_optim[::-1] ) 
plt.xlabel( "$\epsilon$" )  
plt.ylabel( "Time in sec" ) 
plt.legend( loc = "upper right" )
plt.yscale( 'log' )                                                                                                                                                                                                                                                               
plt.savefig( image_folder_path + "/Timeplot_exact_vs_iterative_inversion2.pdf", format = 'pdf' ) 
plt.show()                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        